# Exercise 1 · Check every reply against a schema

You will tighten two rules so that every bad reply fails.

**Goal:** Tighten two schema rules so every bad reply fails.

**Why it matters:** A bad reply that passes gets saved as bad data.

**From the slides:** Slide 4, "A schema guarantees shape, not content; it can cost accuracy"; Slide 5, "Every field is required, and only the amount may be null"; Slide 7, "Validate every reply in your own code anyway".

**Before you start:** You do not write any code. To run a cell, click it and press Shift+Enter. A *schema* is the list of rules for a reply's shape. A *list* is items in square brackets: `["a", "b"]`.

**What the check measures:** your two edits, and that only replies 1 and 4 pass.

**Time:** about 15 minutes.

**Done when:** the last cell prints `Done:`.

## The situation

You order dinner on a food delivery app. When something goes wrong, you send the app a message, such as "My pizza came an hour late." The app has an AI model turn each message into JSON (text with named fields): `order_id`, `issue`, `refund_amount`, `urgent`. A teammate wrote the rules; two are too loose.

## Step 1 · Check with the teammate's rules

Run the given cell below: 6 sample replies (simulated), the teammate's rules and a checker.

Expect `4 of 6 replies pass.` Two of those are bad: reply 2's amount is text, `"about 20 dollars"`; reply 3 has no `urgent`.

In [ ]:
# Given: 6 sample replies (simulated), the teammate's rules, and the checker
import json

import jsonschema

REPLIES = {
    "reply 1": '{"order_id": "A1042", "issue": "late", "refund_amount": null, "urgent": false}',
    "reply 2": '{"order_id": "A3307", "issue": "wrong_item", "refund_amount": "about 20 dollars", "urgent": false}',
    "reply 3": '{"order_id": "A1185", "issue": "late", "refund_amount": null}',
    "reply 4": '{"order_id": "A2210", "issue": "damaged", "refund_amount": 35.5, "urgent": true}',
    "reply 5": '{"order_id": "A2093", "issue": "broken", "refund_amount": 12, "urgent": true}',
    "reply 6": "Sorry, I can't help with that request.",
}
REQUIRED = ["order_id", "issue", "refund_amount"]  # the teammate's rules
AMOUNT_TYPES = ["number", "string", "null"]


def verdict(text):
    # Pass or fail one reply; never change it
    schema = {
        "required": REQUIRED,
        "properties": {
            "issue": {"enum": ["late", "damaged", "wrong_item", "other"]},
            "refund_amount": {"type": AMOUNT_TYPES},
            "urgent": {"type": "boolean"},
        },
    }
    try:
        jsonschema.validate(json.loads(text), schema)
        return "passes"
    except json.JSONDecodeError:
        return "fails: it is not JSON"
    except jsonschema.ValidationError as error:
        return "fails: " + error.message.replace("is a required property", "is missing")
    except Exception:
        return "cannot check: Step 2 or 3 has a typo"


def check_all():
    for name, text in REPLIES.items():
        print(f"{name}: {verdict(text)}")
    print([verdict(t) for t in REPLIES.values()].count("passes"), "of 6 replies pass.")


check_all()

## Step 2 · Require the urgent field

On the `# TODO` line, add `"urgent"` to the list. Run the cell.

Expect `reply 3: fails: 'urgent' is missing`.

<details><summary>Hint</summary>

After `"refund_amount"`, type a comma and `"urgent"`, quotes included.

</details>

<details><summary>Show the answer</summary>

`REQUIRED = ["order_id", "issue", "refund_amount", "urgent"]`

Why: every field is required (Slide 5).

</details>

In [ ]:
# Step 2: the fields every reply must have
REQUIRED = ["order_id", "issue", "refund_amount", "urgent"]
print("reply 3:", verdict(REPLIES["reply 3"]))

## Step 3 · Stop text amounts

`"string"` means text. On the `# TODO` line, delete `"string"` and its comma. Run the cell.

Expect `reply 2: fails: 'about 20 dollars' is not of type 'number', 'null'`.

<details><summary>Hint</summary>

Keep `"number"` and `"null"`, one comma between them.

</details>

<details><summary>Show the answer</summary>

`AMOUNT_TYPES = ["number", "null"]`

Why: the app cannot pay back "about 20 dollars"; it needs a number or null (Slide 5).

</details>

In [ ]:
# Step 3: the kinds of value refund_amount may hold
AMOUNT_TYPES = ["number", "null"]
print("reply 2:", verdict(REPLIES["reply 2"]))

## Step 4 · Check every reply again

Run the cell. Expect `2 of 6 replies pass.`: only replies 1 and 4.

In [ ]:
# Step 4: check every reply again, with your rules
check_all()

## Check your work

Run the cell. If it prints `Not yet:`, redo the step it names.

In [ ]:
# Check your work: both rules, then every reply
try:
    if sorted(REQUIRED) != ["issue", "order_id", "refund_amount", "urgent"]:
        found = 'REQUIRED should list the four fields. Redo Step 2 and run its cell.'
    elif sorted(AMOUNT_TYPES) != ["null", "number"]:
        found = 'AMOUNT_TYPES should be ["number", "null"]. Redo Step 3 and run its cell.'
    elif [verdict(t) == "passes" for t in REPLIES.values()] != [True, False, False, True, False, False]:
        found = "the Step 1 cell was changed. Undo that and run every cell."
    else:
        found = ""
except Exception:
    found = "a cell above has not run. Run every cell from the top."
print(f"Not yet: {found}" if found else "Done: replies 1 and 4 pass, the other four fail, none fixed.")

## What you learned

- A schema checks only the rules you write; a loose rule lets bad replies through (Slides 4 and 5).
- Check every reply in your own code; a bad one is a failure, never fixed (Slide 7).